# Experiment 4 · Cross-dataset transfer, traditional baselines

How well do the traditional baselines generalise **out of distribution**? A model trained
on one dataset (source **A**) is evaluated, with no retraining, on the test split of every
other dataset (target **B**). Sweeping the four datasets against each other fills a 4x4
source-to-target matrix; the in-domain diagonal is the Experiment 1 result and is kept as a
sanity check that the transfer harness reproduces it.

Every model is evaluated **box-conditioned**, receiving the same tight oracle box it was
trained with as a fourth input channel. This keeps the comparison matched against the
promptable foundation models in the sibling `foundation/` folder: both families see the
same prompt, so a difference between them is a difference in the model, not in what it was
told about the nodule's location.


In [ ]:
import os
from pathlib import Path
while not (Path.cwd() / 'pyproject.toml').exists() and Path.cwd() != Path.cwd().parent:
    os.chdir('..')
print('repo root:', Path.cwd())

In [ ]:
# Check the inputs this experiment needs before doing anything slow. A missing
# dataset here means an unrun (or unplaced) setup step, not a bug in the experiment.
from pathlib import Path

REQUIRED = ['ddti', 'tn3k', 'thyroidxl', 'stanford_aimi']
GATED = {'thyroidxl', 'stanford_aimi'}

missing = [d for d in REQUIRED
           if not (Path('data/processed') / d / 'images').is_dir()
           or not (Path('data/splits') / f'{d}_test.csv').exists()]
if missing:
    print('Missing preprocessed data or splits for:', ', '.join(missing))
    for d in missing:
        if d in GATED:
            print(f'  {d:14s} access-gated -> place your approved copy first, see '
                  f'00_setup/01_place_gated_datasets.ipynb')
        else:
            print(f'  {d:14s} open -> download it with 00_setup/00_get_open_datasets.ipynb')
    print('Then run 00_setup/02_preprocess.ipynb and 00_setup/03_make_splits.ipynb.')
    print('\nYou can still run this experiment on whichever datasets ARE present '
          'by restricting the --dataset argument below.')
else:
    print('All four datasets are preprocessed and split.')


## Prerequisites

- Datasets downloaded and preprocessed to 512x512 PNGs via `00_setup/` (populates
  `data/processed/` and `data/splits/`).
- **Experiment 1 checkpoints.** This experiment trains nothing. It loads each source
  dataset's box-conditioned checkpoint from
  `experiments/exp1_fullsupervised/traditional_boxcond/` and runs inference on the other
  datasets' test splits. Run Experiment 1 first.
- For the nnU-Net leg you additionally need `nnUNetv2_predict` on your PATH and the
  box-conditioned nnU-Net datasets built by `convert_to_nnunet_boxcond.py`.


## Run

Two streams write rows in the same schema, so they merge into one table.

**Stream A - U-Net and TransUNet.** `run.py` resolves the best-LR checkpoint for each
`--model` / `--src` pair and evaluates it on `--tgt`. Omitting `--src` / `--tgt` sweeps all
of them; the diagonal is kept unless you pass `--skip_diagonal`. This is inference only,
so the full 2 x 4 x 4 sweep is a matter of hours, not days.

**Stream B - nnU-Net.** nnU-Net predicts through its own CLI, so the cross-dataset cells
are produced by running `nnUNetv2_predict` with the source-trained model over the target's
`imagesTs`, then scoring the predictions with `eval_nnunet_cross.py`.


In [ ]:
# Stream A: box-conditioned U-Net and TransUNet, full 4x4 matrix (inference only).
!python experiments/exp4_crossdataset/traditional/run.py --model all


### Stream B - nnU-Net cross-dataset cells

For each source-to-target pair, predict with the source-trained fold-0 model over the
target's test images, then score. `--pred_dir` is the directory `nnUNetv2_predict` wrote.


In [ ]:
# One source -> target cell, shown explicitly. Repeat for the 12 off-diagonal pairs.
# Dataset ids: 011=DDTI, 012=TN3K, 013=ThyroidXL, 014=Stanford AIMI.
#
# export nnUNet_raw=data/nnunet_raw
# export nnUNet_preprocessed=experiments/exp1_fullsupervised/traditional_boxcond/results/nnunet/preprocessed
# export nnUNet_results=experiments/exp1_fullsupervised/traditional_boxcond/results/nnunet/results
#
# nnUNetv2_predict -i data/nnunet_raw/Dataset012_TN3K_boxcond/imagesTs \
#     -o experiments/exp4_crossdataset/traditional/preds/src_ddti_tgt_tn3k \
#     -d 011 -c 2d -f 0                      # <- model trained on DDTI (011)
#
# !python experiments/exp4_crossdataset/traditional/eval_nnunet_cross.py \
#     --src ddti --tgt tn3k \
#     --pred_dir experiments/exp4_crossdataset/traditional/preds/src_ddti_tgt_tn3k


## Results

The per-transfer metrics are consolidated in
`results/boxcond_cnn_crossdataset_master.csv`, the table behind the cross-dataset results.

In [ ]:
import pandas as pd

df = pd.read_csv('experiments/exp4_crossdataset/traditional/results/boxcond_cnn_crossdataset_master.csv')
df